In [121]:
!pip install --upgrade transformers datasets evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 65.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.1
    Uninstalling transformers-4.48.1:
      Successfully uninstalled transformers-4.48.1
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import re
import random
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score,
)
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    ModernBertModel,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from datasets import (
    load_dataset, 
    ClassLabel, 
)
import evaluate

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [119]:
MODEL_NAME = "distilbert-base-uncased"
# MODEL_NAME = "answerdotai/ModernBERT-base"

TEST_SIZE = 0.2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 20
BATCH_SIZE = 16
DROPOUT_RATE = 0.2
EXTRA_LAYERS = []

DATA_PATH = "data/sentence_sets_trimmed.csv"
DATA_ENCODING = "ISO-8859-1"

LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

DEGENDER_ENABLED = True
STRATIFY_ENABLED = True
BALANCE_ENABLED = True

DEGENDER_LEVEL = "enhanced"
BALANCE_TYPE = "over"

MAX_LENGTH = 512

LOG_DIR = "~/scratch/log"

OUTPUT_DIR = "~/scratch"
OUTPUT_NAME = f"{MODEL_NAME}-finetuned-nlp-letters-{TEXT_COLUMN}"
OUTPUT_NAME += f"-degendered" if DEGENDER_ENABLED else ""
OUTPUT_NAME += f"-stratified" if STRATIFY_ENABLED else ""
OUTPUT_NAME += f"-balanced-{BALANCE_TYPE}" if BALANCE_ENABLED else ""

In [120]:
if torch.cuda.is_available():
    print("CUDA is available. Using GPU:", torch.cuda.get_device_name(0))
    print("Number of GPUs available:", torch.cuda.device_count())
else:
    print("CUDA is not available. Running on CPU.")

CUDA is available. Using GPU: Tesla V100-PCIE-16GB
Number of GPUs available: 2


In [121]:
class LettersBERTModule(nn.Module):
    def __init__(self, model_name, num_labels, extra_layers=[], dropout_rate=0.2, class_weights=None, cls_or_mean="cls"):
        super().__init__()

        self.config = AutoConfig.from_pretrained(model_name)
        self.config.num_labels = num_labels
        self.config.class_weights = class_weights
        self.config.cls_or_mean = cls_or_mean

        # BERT transformer
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        
        # Pre-classifier
        self.pre_classifier = nn.Linear(self.config.hidden_size, self.config.hidden_size)
        self.pre_classifier_act = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # Extra dense layers with dropout and batch normalization
        self.extra_layers = nn.ModuleList()
        in_features = self.config.hidden_size
        for features in extra_layers:
            self.extra_layers.append(nn.Sequential(
                nn.Linear(in_features, features),
#                 nn.BatchNorm1d(features),
                nn.ReLU(),
                nn.Dropout(p=dropout_rate)
            ))
            in_features = features        
        
        # Classifier        
        self.classifier = nn.Linear(in_features, self.config.num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Output from transformer
        if self.config.cls_or_mean == "mean":
            output = torch.mean(outputs.last_hidden_state, dim=1)
        else:
            output = outputs.last_hidden_state[:, 0, :]

        # Feed forward 
        x = self.pre_classifier(output)
        x = self.pre_classifier_act(x)
        x = self.dropout(x)

        for layer in self.extra_layers:
            x = layer(x)        
        
        # Logits and loss
        logits = self.classifier(x)
        loss = None

        if labels is not None:
            if self.config.class_weights:
                loss_func = nn.CrossEntropyLoss(
                    weight=torch.tensor(self.config.class_weights, dtype=torch.float32).to(input_ids.device)
                )
            else:
                loss_func = nn.CrossEntropyLoss(weight=self.config.class_weights)
            loss = loss_func(logits, labels)

        return {"loss": loss, "logits": logits}

In [122]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [123]:
def degender(text, level="original"):
    if level == "enhanced":
        titles = r"\b(?:mr|mrs|ms|miss|mister|sir|madam)\b"
        nouns = r"\b(?:man|men|woman|women|gentleman|lady|boy|boys|girl|girls)(?:'s)?\b"
        pronouns = r"\b(?:he|him|his|she|her|hers|himself|herself)\b"
    else:
        titles = r"\b(?:mr|mrs|ms|miss|mister)\b"
        nouns = r"\b(?:man|men|woman|women|gentleman|lady)(?:'s)?\b"
        pronouns = r"\b(?:he|him|his|she|her|hers)\b"

    titles_regex = re.compile(titles, flags=re.IGNORECASE)
    nouns_regex = re.compile(nouns, flags=re.IGNORECASE)
    pronouns_regex = re.compile(pronouns, flags=re.IGNORECASE)

    text = titles_regex.sub("mx", text)
    text = nouns_regex.sub("person", text)
    text = pronouns_regex.sub("they", text)

    return text

In [124]:
def preprocess(data):
    texts = data[TEXT_COLUMN]

    if DEGENDER_ENABLED:
        texts = [degender(t.lower(), DEGENDER_LEVEL) for t in texts]

    tokenized = tokenizer(
        texts, 
        truncation=True, 
        padding=True
    )

    tokenized["labels"] = data[LABEL_COLUMN]

    return tokenized

In [125]:
def preprocess_nodegender(data):
    texts = data[TEXT_COLUMN]

    tokenized = tokenizer(
        texts, 
        truncation=True, 
        padding=True
    )

    tokenized["labels"] = data[LABEL_COLUMN]

    return tokenized

In [126]:
def balance(dataset, key="labels", type="under"):
    labels = dataset[key]
    labels_unique = list(set(labels))
    labels_count = [labels.count(l) for l in labels_unique]

    min_count = min(labels_count)
    max_count = max(labels_count)

    indices = []

    if type.lower() == "over":
        # Oversample: increases each class to the maximum count
        for label in labels_unique:
            label_indices = [i for i, l in enumerate(labels) if l == label]
            n = len(label_indices)
            multiplier = max_count // n
            remainder = max_count % n

            # Duplicate and add a random subset for the remainder
            indices.extend(label_indices * multiplier)
            if remainder > 0:
                indices.extend(random.sample(label_indices, remainder))

    elif type.lower() == "under":
        # Undersample: reduces each class to the minimum count
        for label in labels_unique:
            label_indices = [i for i, l in enumerate(labels) if l == label]
            sampled_indices = random.sample(label_indices, min_count)
            indices.extend(sampled_indices)

    else:
        # Randomize the indices
        indices = list(range(len(labels)))        
    
    random.shuffle(indices)
    balanced_dataset = dataset.select(indices)
    
    # Print breakdown of class distribution in the balanced dataset
    print("Balanced dataset class distribution:", Counter(balanced_dataset[key]))
    
    return balanced_dataset

In [127]:
# Load the data
dataset = load_dataset("csv", data_files=DATA_PATH, encoding=DATA_ENCODING)

# Recast labels as class features
unique_labels = dataset["train"].unique(LABEL_COLUMN)

features = dataset["train"].features
features[LABEL_COLUMN] = ClassLabel(names=unique_labels)

dataset = dataset.cast(features)

dataset = dataset["train"].train_test_split(
    test_size=TEST_SIZE,
    stratify_by_column=LABEL_COLUMN if STRATIFY_ENABLED else None,
    seed=100,
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

if BALANCE_ENABLED:
    train_dataset = balance(train_dataset, key=LABEL_COLUMN, type=BALANCE_TYPE)

Balanced dataset class distribution: Counter({0: 1879, 1: 1879})


In [128]:
print([degender(t.lower()) for t in dataset["train"][TEXT_COLUMN][:10]])

["first_name middle_name paz de araujo   a fourth   year medical student at the university of colorado health sciences center   currently applying for a position in your anesthesiology residency program  * i had the opportunity to work with first_name during clinical clerkships at children's hospital of colorado in both they third and fourth year of medical school  * first_name excelled in all phases of they clinical responsibilities during they pediatric anesthesiology clerkship  * i often found first_name remaining in the or later than usual to observe and participate in interesting cases taking place later in the first_name  * airway and intravenous access skills were outstanding and significantly above the standard expected for first_name's current level of training  * first_name displayed a comfort and keen ability to interact and connect with pediatric patients and their families  * first_name's commitment to public health and research are evident with they participation in the s

In [129]:
train_dataset = train_dataset.map(preprocess_nodegender, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

In [130]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
confusion_metric = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    precision = precision_metric.compute(predictions=preds, references=labels, average="macro")
    recall = recall_metric.compute(predictions=preds, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")

    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    
    cr = classification_report(labels, preds, target_names=train_dataset.features[LABEL_COLUMN].names)
    cm = confusion_metric.compute(predictions=preds, references=labels)

    print("Confusion Matrix:\n", cm["confusion_matrix"])
    print("Classification Report:\n", cr)
    print("MCC:", mcc)
    print("Balanced Accuracy:", bal_acc)

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"],
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
    }

In [131]:
# Class weight balancing
class_weights = None
if BALANCE_ENABLED and BALANCE_TYPE == "weights":
    class_weights = compute_class_weight(
        "balanced", 
        classes=np.unique(train_dataset[LABEL_COLUMN]), 
        y=train_dataset[LABEL_COLUMN]
    ).tolist()

In [132]:
model = LettersBERTModule(
    model_name=MODEL_NAME, 
    num_labels=len(train_dataset.features[LABEL_COLUMN].names),
    extra_layers=EXTRA_LAYERS,
    dropout_rate=DROPOUT_RATE,
    class_weights=class_weights
)

In [133]:
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, 
#     num_labels=len(train_dataset.features[LABEL_COLUMN].names)
# )

In [134]:
# model = ModernBertModel.from_pretrained("answerdotai/ModernBERT-base")

In [135]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LettersBERTModule(
  (transformer): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1)

In [136]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/{OUTPUT_NAME}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir=LOG_DIR,
    logging_steps=50,
    no_cuda=not torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_3540036/4141954113.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [137]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy
1,0.007200,1.649659,0.286149,0.642530,0.501064,0.223950,0.024627,0.501064
2,0.002700,2.384722,0.289193,0.542638,0.501582,0.230114,0.016424,0.501582
3,0.001900,1.293692,0.328767,0.516694,0.506702,0.302284,0.021154,0.506702
4,0.001800,1.638075,0.307458,0.532540,0.506298,0.263796,0.028630,0.506298
5,0.001200,1.135089,0.382040,0.531378,0.523006,0.375545,0.053736,0.523006
6,0.001200,1.485612,0.333333,0.528224,0.511503,0.307456,0.036037,0.511503
7,0.001000,1.610756,0.324201,0.524152,0.508340,0.293480,0.028385,0.508340
8,0.001400,1.553751,0.333333,0.528224,0.511503,0.307456,0.036037,0.511503
9,0.001800,0.927397,0.410959,0.528076,0.525509,0.410035,0.053523,0.525509
10,0.000600,2.313749,0.286149,0.642530,0.501064,0.223950,0.024627,0.501064


Confusion Matrix:
 [[  1 469]
 [  0 187]]
Classification Report:
               precision    recall  f1-score   support

        male       1.00      0.00      0.00       470
      female       0.29      1.00      0.44       187

    accuracy                           0.29       657
   macro avg       0.64      0.50      0.22       657
weighted avg       0.80      0.29      0.13       657

MCC: 0.024627478840987982
Balanced Accuracy: 0.5010638297872341


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Confusion Matrix:
 [[  4 466]
 [  1 186]]
Classification Report:
               precision    recall  f1-score   support

        male       0.80      0.01      0.02       470
      female       0.29      0.99      0.44       187

    accuracy                           0.29       657
   macro avg       0.54      0.50      0.23       657
weighted avg       0.65      0.29      0.14       657

MCC: 0.01642352075461342
Balanced Accuracy: 0.5015815223574923


...

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Confusion Matrix:
 [[  1 469]
 [  1 186]]
Classification Report:
               precision    recall  f1-score   support

        male       0.50      0.00      0.00       470
      female       0.28      0.99      0.44       187

    accuracy                           0.28       657
   macro avg       0.39      0.50      0.22       657
weighted avg       0.44      0.28      0.13       657

MCC: -0.026374306899221333
Balanced Accuracy: 0.49839003299579016


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Confusion Matrix:
 [[  1 469]
 [  1 186]]
Classification Report:
               precision    recall  f1-score   support

        male       0.50      0.00      0.00       470
      female       0.28      0.99      0.44       187

    accuracy                           0.28       657
   macro avg       0.39      0.50      0.22       657
weighted avg       0.44      0.28      0.13       657

MCC: -0.026374306899221333
Balanced Accuracy: 0.49839003299579016


TrainOutput(global_step=2360, training_loss=0.005140024402350981, metrics={'train_runtime': 837.1361, 'train_samples_per_second': 89.782, 'train_steps_per_second': 2.819, 'total_flos': 0.0, 'train_loss': 0.005140024402350981, 'epoch': 20.0})

In [138]:
trainer.evaluate()

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Confusion Matrix:
 [[122 348]
 [ 39 148]]
Classification Report:
               precision    recall  f1-score   support

        male       0.76      0.26      0.39       470
      female       0.30      0.79      0.43       187

    accuracy                           0.41       657
   macro avg       0.53      0.53      0.41       657
weighted avg       0.63      0.41      0.40       657

MCC: 0.05352320303878683
Balanced Accuracy: 0.5255091591762431


{'eval_loss': 0.9273973107337952,
 'eval_accuracy': 0.410958904109589,
 'eval_precision': 0.5280755359647366,
 'eval_recall': 0.5255091591762431,
 'eval_f1': 0.4100349673877482,
 'eval_mcc': 0.05352320303878683,
 'eval_balanced_accuracy': 0.5255091591762431,
 'eval_runtime': 2.6621,
 'eval_samples_per_second': 246.794,
 'eval_steps_per_second': 7.888,
 'epoch': 20.0}